# Introduction to Pandas and DataFrame

Lina Khelfet and Stéphane Bourg

This notebook is an adaption from pandas user guide, see [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html).

## Pandas

**Pandas** is an open source, BSD-licensed library providing high-performance, easy-to-use data structures and data analysis tools for the Python programming language.

Data scientists often work with data stored in table formats like `.csv`, `.tsv`, or `.xlsx`

In [ ]:
import numpy as np
import pandas as pd

### Data Structure

Creating a **Series** by passing a list of values, letting pandas create a default `RangeIndex`.

In [ ]:
s = pd.Series([1, 3, 5, np.nan, 6, 8]) # np.nan is used to generate a missing  value (not a number) into the data
s

Creating a DataFrame from a dictionary

In [ ]:
data = {
  "calories": [420, 380, 390],
  "duration": [50, 40, 45]
}

df_dic = pd.DataFrame(data)
df_dic

 ### The Iris Dataset

 This data sets consists of 50 samples from each of 3 different types of irises (Setosa, Versicolour, and Virginica).

In [ ]:
import sklearn
from sklearn import datasets

iris = datasets.load_iris()
iris

In [ ]:
print(iris.DESCR)

**Features are the input variables used to predict an outcome, while target is the output variable you want to predict.**

Here, four features were measured from each sample: the **length** and the **width** of the *sepals* and *petals*, in centimeters. Based on the combination of these four features, we aim to predict the specie of Iris (target):
- 0: Setosa
- 1: Versicolour
- 2: Virginica

An **array** is an ordered list of values stored in a single container. It’s used to collect data of the **same type** (numbers, strings, measurements, etc.) in a **compact, efficient way**.

In [ ]:
# features
X = iris.data # A 2D array that contains the input data.

# targeted variable
y = iris.target # A 1D array that contains the labels or categories for each data point.

In [ ]:
type(X), type(y)

In [ ]:
X.shape, y.shape

In [ ]:
X

### Convert numpy array into DataFrame

In [ ]:
df = pd.DataFrame(X, columns = iris.feature_names)
df['target'] = pd.Series(y) # handle the target separately as it is an array
df

### Exploratory Data Analysis (EDA)

EDA is an important step in data analysis which focuses on understanding content of a dataset as well as its patterns, trends and relationships through statistical tools and visualisations. 

#### Quick look

Use `DataFrame.head()` and `DataFrame.tail()` to view the top and bottom rows of the frame respectively.

In [ ]:
df.head() # display the first 5 rows by default

In [ ]:
df.tail(10) # display the last 10 rows

Check dimensions

In [ ]:
df.shape

Check columns names and types

In [ ]:
df.columns

Check columns types

In [ ]:
df.dtypes

`info()` helps us to understand the dataset by showing the number of records in each column, type of data, whether any values are missing and how much memory the dataset uses.

In [ ]:
df.info()

`describe()` gives a statistical summary of the DataFrame showing values like count, mean, standard deviation, minimum and quartiles for each numerical column. It helps in summarizing the central tendency and spread of the data.

*NB: each statistic (mean, standard deviation, min, max, etc.) can also be calculated individually using their respective functions (e.g., `.mean()`, `.std()`).*

In [ ]:
df.describe()

`sample()` returns a specified number of random rows.

In [ ]:
df.sample(5)

To return a series containing the frequency of each distinct row in the Dataframe, we can use the `value_counts` method. 

In [ ]:
df["target"].value_counts()

#### Indexing and retrieving data

A DataFrame can be indexed in a few different ways. To get a single column, you can use either `DataFrame['Name']` or `DataFrame.Name` constructions. 

In [ ]:
df["petal width (cm)"] # display the petal width column only

In [ ]:
df[0:3] # first three rows

**Exercise: what is the average petal width of irises?**

In [ ]:
df["petal width (cm)"].mean().round(4) # round to 4 decimal places

Selecting by label or position (index)

In [ ]:
df.loc[0, "sepal length (cm)"] # selecting a single row and column label returns a scalar

In [ ]:
df.loc[0:5, ["sepal length (cm)", "petal length (cm)"]] # selecting multiple rows and columns with label slicing

In [ ]:
df.iloc[3:5, 0:2] # selecting by position: rows 3 to 4 and columns 0 to 1

In [ ]:
df.iloc[1, 1] # selecting a single value by position

**Boolean indexing** with one column is also very convenient. The syntax is `df[C(df['Name'])]`, where `C` is some logical condition that is checked for each element of the Name column. The result of such indexing is the DataFrame consisting only of rows that satisfy the `C` condition on the `Name` column.

In [ ]:
df[(df["sepal width (cm)"] > 3.5)] # boolean indexing using 3.5 as threshold as a condition

`isin()` method checks if the Dataframe contains the specified value(s)

In [ ]:
selected_targets = [0, 2]

df[df['target'].isin(selected_targets)]

`iterrows()` enables to iterate over rows of a DataFrame as (index, Series) pairs.

In [ ]:
for index, row in df.head().iterrows():
    print(f"Row {index}: Sepal length = {row['sepal length (cm)']}, Target = {row['target']}") # iterate over rows

#### Sorting

A DataFrame can be sorted by the value of one of the variables (i.e columns). For example, we can sort by `sepal length` (use ascending=False to sort in descending order):

In [ ]:
df.sort_values(by="sepal length (cm)", ascending=False, inplace=True) # inplace=True modifies the original DataFrame
df.head()

**Exercise: sort by sepal length (cm) and petal length (cm), in descending and ascending order respectively**

In [ ]:
df.sort_values(by=["sepal length (cm)", "petal length (cm)"], ascending=[False, True], inplace=True)
df.head()

Pandas first sorts the entire DataFrame by sepal length (cm) in descending order. Only within each group of equal sepal length (cm) values does it sort petal length (cm) in ascending order.

#### Groupby

`groupby` is a powerful tool used to **split a DataFrame into groups** based on one or more columns, allowing for efficient data analysis and aggregation. This can be used to group large amounts of data and compute operations on these groups.

It follows a "split-apply-combine" strategy, where data is divided into groups, a function is applied to each group, and the results are combined into a new DataFrame.

In [ ]:
df.groupby('target')['sepal length (cm)'].mean()

In [ ]:
df.groupby('target')[['sepal length (cm)', 'petal length (cm)']].agg(['mean', 'max'])

Example of transform with rank lambda function

In [ ]:
df['rank_sepal'] = df.groupby('target')['sepal length (cm)'].transform(lambda x: x.rank(ascending=False))
df.head()

### Apply a function to the DataFrame

`iterrows` can also be used to apply a function to specific rows, but it is less efficient for big DataFrames.

In [ ]:
def sepal_category(row):
    if row['sepal length (cm)'] > 5.5:
        return 'long'
    else:
        return 'short'

# Apply the function to each row and collect results
categories = []
for index, row in df.iterrows():
    categories.append(sepal_category(row))

# Add the results as a new column
df['sepal_category'] = categories

df['sepal_category'].value_counts()

`map` is used to apply a function or mapping to each element of a **single Series**. It is often used to transform values based on a correspondence, such as mapping values from another Series or a dictionary.

In [ ]:
target_map = {0: 'setosa', 1: 'versicolor', 2: 'virginica'} # mapping from a dictionary
df['species'] = df['target'].map(target_map)
df['species'].value_counts()

In [ ]:
df.head()

`apply` can be used on a **Series** or a **DataFrame** and is more flexible. The function passed as an argument typically works on rows/columns. 

In [ ]:
df['sepal_length_category'] = df['sepal length (cm)'].apply(
    lambda x: 'long' if x > 5.5 else 'short'
)

df['sepal_length_category'].value_counts()

The `applymap()` method only works on a pandas Dataframe where a function is applied to every element individually. The function is typically used for elementwise operations (DEPRECATED). 

In [ ]:
numeric_cols = df.select_dtypes(include='number')  # select numeric columns
df[numeric_cols.columns] = numeric_cols.applymap(lambda x: round(x, 0)) # round all numeric values to the nearest integer
df[numeric_cols.columns]

| Function   | Works on           | Example use case                                               |
| ---------- | ------------------ | -------------------------------------------------------------- |
| `iterrow`  | Series             | Transform values in specific rows of a column                  |
| `map`      | Series             | Transform values in a column                                   |
| `apply`    | Series / DataFrame | Row-wise or column-wise operations, or complex transformations |
| `applymap` | DataFrame          | Element-wise operation for **all cells** in a DataFrame        |